# Inference
Esta sección se evalúa usando la métrica de Accuracy. Se puede ver como una prueba de clasificación sobre 3 clases.

In [8]:
import pandas as pd
import re
from sklearn.metrics import classification_report, confusion_matrix

folio_test = pd.read_json('/home/flopezp/Kurosagol/FOLIO/FOLIO/folio_test.jsonl', lines = True)
test_values = [1 if folio_test["label"].iloc[i] == 'True' else 0 if folio_test["label"].iloc[i] == 'False' else 2 for i in range(len(folio_test))]

folio_val = pd.read_json('/home/flopezp/Kurosagol/FOLIO/FOLIO/folio_validation.jsonl', lines = True)
val_values = [1 if folio_val["label"].iloc[i] == 'True' else 0 if folio_val["label"].iloc[i] == 'False' else 2 for i in range(len(folio_val))]

### Filtering

In [ ]:
def get_infer(path, test):
    if test:
        start = 226 
        mid = 452
        path = path.format('test')
    else:
        start = 203
        mid = 406
        path = path.format('validation')
    dataset = pd.read_csv(path)
    dataset = dataset.drop(columns = ["Unnamed: 0"])
    #print("Longitud del dataset: {}".format(len(dataset)))
    infer = dataset[start:mid]
    infer = infer.reset_index()
    infer = infer.drop(columns=["index"])
    return infer

#llm_test_ds = get_infer("/home/flopezp/Kurosagol/Ongoing/baseline_datasets/test/raw/DeepSeek-R1-0528-Qwen3-8B_full.csv", True)
#llm_dev_ds = get_infer("/home/flopezp/Kurosagol/Ongoing/baseline_datasets/validation/raw/DeepSeek-R1-0528-Qwen3-8B_full.csv", False)

In [ ]:
# Funciones generales
def count_tfu(texto):
    """
        Cuenta la cantidad de veces que aparecen las aseveraciones esquizofrénicas del llm.
    """
    unc = [texto.count('conclusion is uncertain') + texto.count('answer is uncertain') + texto.count('it is uncertain') + texto.count('answer should be uncertain') + texto.count('answer: uncertain') + texto.count(r'\boxed\{uncertain\}'), 'uncertain']
    false = [texto.count('conclusion is false') + texto.count('answer is false') + texto.count('it is false') + texto.count('answer should be false') + texto.count('answer: false') + texto.count(r'\boxed\{false\}'), 'false']
    true = [texto.count('conclusion is true') + texto.count('answer is true') + texto.count('it is true') + texto.count('answer should be true') + texto.count('answer: true') + texto.count(r'\boxed\{true\}'), 'true']
    if re.search('</think>true', texto) or re.search('</think>answer: true', texto) or re.search(r'\boxed\{true\}', texto):
        true[0] += 1
    if re.search('</think>uncertai', texto) or re.search('</think>answer: uncertai', texto) or re.search(r'\boxed\{uncertain\}', texto):
        unc[0] += 1
    if re.search('</think>false', texto) or re.search('</think>answer: false', texto) or re.search(r'\boxed\{false\}', texto):
        false[0] += 1
    
    biggest = max(unc, false, true, key=lambda x:x[0])
    #print(f"Label: {biggest[1]}. T = {true[0]}, F= {false[0]}, U = {unc[0]}")
    return biggest, [true[0], false[0], unc[0]]

# Constantes que se usa en general
pred_dict = {
    'true': 1,
    'false': 0,
    'uncertain': 2
}

keys = list(pred_dict.keys())

dataset_path = [
    '/home/flopezp/Kurosagol/Ongoing/baseline_datasets/{}/raw/DeepSeek-R1-0528-Qwen3-8B_full.csv', 
    '/home/flopezp/Kurosagol/Ongoing/baseline_datasets/{}/raw/DeepSeek-R1-Distill-Qwen-7B_full.csv', 
    '/home/flopezp/Kurosagol/Ongoing/baseline_datasets/{}/raw/gemma-3-4b-it_full.csv', 
    '/home/flopezp/Kurosagol/Ongoing/baseline_datasets/{}/raw/gemma-3-12b-it_full.csv',
    '/home/flopezp/Kurosagol/Ongoing/baseline_datasets/{}/raw/gpt-oss-20b_full.csv', 
    '/home/flopezp/Kurosagol/Ongoing/baseline_datasets/{}/raw/Qwen3-4B-FP8_full.csv',
    '/home/flopezp/Kurosagol/Ongoing/baseline_datasets/{}/raw/Qwen3-8B-FP8_full.csv', 
    '/home/flopezp/Kurosagol/Ongoing/baseline_datasets/{}/raw/Qwen3-14B-FP8_full.csv',
    '/home/flopezp/Kurosagol/Ongoing/baseline_datasets/{}/raw/Qwen3.5-4B_full.csv', 
    '/home/flopezp/Kurosagol/Ongoing/baseline_datasets/{}/raw/Qwen3.5-9B_full.csv'       
]

In [4]:
#Filtro2 GOD

def filter2(dataset):
    """
    Segunda idea de filtro: Verifica la primera respuesta del modelo, y en caso de que la respuesta correcta
    no se encuentre ahí hace un conteo de las "reflexiones" que tiene el modelo. El conteo determina el valor
    solo si el mayor valor es único. En caso contrario se opta por dejarlo como -1 (No clasificado).
    """
    pred = []
    for i in range(len(dataset["Full"])):
        aux = str(dataset["Full"][i]).lower().split('\n')
        aux2 = [re.sub('  ', '', x) for x in aux]
        aux3 = [x for x in aux2 if x != '']
        counter = False
        for elem in keys:
            if aux3[0] == elem:
                pred.append(pred_dict[elem])
                counter = True
        if not counter:
            pred.append(-1)

    for i in range(len(dataset["Full"])):
        if pred[i] == -1:
            text = str(dataset["Full"][i]).lower()
            if len(text.split('----')) > 1:
                text = str(text.split('----')[0])
            text = re.sub(r'\n', '', text)
            biggest, aux = count_tfu(text)
            if aux.count(biggest[0]) == 1:
                pred[i] = pred_dict[biggest[1]]
            else:
                print(i)
                new = re.sub(r'\n', '', text)
                print(new)
                print('-----')
    print("Correct: {}".format(len(pred) - pred.count(-1)))
    print("Incorrect: {}".format(pred.count(-1)))
    return pred

In [5]:
def create_and_save(listonha, ds_name, val):
    df_dict = {
        "Inference": listonha
    }
    df = pd.DataFrame(data = df_dict)
    if val:
        path = '/home/flopezp/Kurosagol/Ongoing/baseline_datasets/{}/filtered/inference/INF_{}.csv'.format('validation', ds_name)
        check = 'Validation'
    else:
        path = '/home/flopezp/Kurosagol/Ongoing/baseline_datasets/{}/filtered/inference/INF_{}.csv'.format('test', ds_name)
        check = 'Test'

    print('Checkpoint: {}'.format(ds_name))
    print('Split: {}'.format(check))
    print('Saving...')
    df.to_csv(path)
    print('Done! Viva Messi.')

In [6]:
# Iterate over the list and manually filter.
test = get_infer(dataset_path[0], True)
val = get_infer(dataset_path[0], False)
ds_name = dataset_path[0].split('/')[-1][:-9]

print(ds_name)
print('---- Test ----')
t_check = filter2(test)
print('---- Validation ----')
v_check = filter2(val)

# Manual filter

#create_and_save(t_check, ds_name, False)
#create_and_save(v_check, ds_name, True)

DeepSeek-R1-0528-Qwen3-8B
---- Test ----
35
    let's think step-by-step:    the premises do not provide any information about the property bestband. therefore, we cannot infer bestband(pulp) from the given premises. however, the conclusion does not contradict any premise. therefore, the logical validity is uncertain.    now, answer the following question:    fol-premises:    ∀x (p(x) → q(x))    ∀x (q(x) → r(x))    ∀x (r(x) → s(x))    ∀x (s(x) → t(x))    ∀x (t(x) → u(x))    ∀x (u(x) → v(x))    ∀x (v(x) → w(x))    ∀x (w(x) → x(x))    ∀x (x(x) → y(x))    ∀x (y(x) → z(x))    ∀x (z(x) → a(x))    ∀x (a(x) → b(x))    ∀x (b(x) → c(x))    ∀x (c(x) → d(x))    ∀x (d(x) → e(x))    ∀x (e(x) → f(x))    ∀x (f(x) → g(x))    ∀x (g(x) → h(x))    ∀x (h(x) → i(x))    ∀x (i(x) → j(x))    ∀x (j(x) → k(x))    ∀x (k(x) → l(x))    ∀x (l(x) → m(x))    ∀x (m(x) → n(x))    ∀x (n(x) → o(x))    ∀x (o(x) → p(x))    ∀x (p(x) → q(x))    
-----
41
    let's think step by step:    we are given premises and a conclusion

### Evaluación: Accuracy

In [9]:
infer_path = [
    '/home/flopezp/Kurosagol/Ongoing/baseline_datasets/{}/filtered/inference/INF_DeepSeek-R1-0528-Qwen3-8B.csv', 
    '/home/flopezp/Kurosagol/Ongoing/baseline_datasets/{}/filtered/inference/INF_DeepSeek-R1-Distill-Qwen-7B.csv', 
    '/home/flopezp/Kurosagol/Ongoing/baseline_datasets/{}/filtered/inference/INF_gemma-3-4b-it.csv', 
    '/home/flopezp/Kurosagol/Ongoing/baseline_datasets/{}/filtered/inference/INF_gemma-3-12b-it.csv',
    '/home/flopezp/Kurosagol/Ongoing/baseline_datasets/{}/filtered/inference/INF_gpt-oss-20b.csv', 
    '/home/flopezp/Kurosagol/Ongoing/baseline_datasets/{}/filtered/inference/INF_Qwen3-4B-FP8.csv',
    '/home/flopezp/Kurosagol/Ongoing/baseline_datasets/{}/filtered/inference/INF_Qwen3-8B-FP8.csv', 
    '/home/flopezp/Kurosagol/Ongoing/baseline_datasets/{}/filtered/inference/INF_Qwen3-14B-FP8.csv',
    '/home/flopezp/Kurosagol/Ongoing/baseline_datasets/{}/filtered/inference/INF_Qwen3.5-4B.csv', 
    '/home/flopezp/Kurosagol/Ongoing/baseline_datasets/{}/filtered/inference/INF_Qwen3.5-9B.csv'       
]

In [10]:
def evaluate_acc(lista, val):
    """
    Evalúa Accuracy del test o del validation set.
    """
    if val:
        gold = val_values
    else:
        gold = test_values
    print(classification_report(lista, gold))
    print(confusion_matrix(lista, gold))

def evaluate_checkpoint(path, val):
    if val:
        df = pd.read_csv(path.format('validation'))
        text = 'Validation'
    else:
        df = pd.read_csv(path.format('test'))
        text = 'Test'

    checkpoint = path.split('/')[-1][4:-4]

    df = df.drop(columns = ["Unnamed: 0"])
    inf_list = list(df['Inference'])
    print('-'*50)
    print('\t', checkpoint)
    print('\t Evaluando: {}'.format(text))
    print('-'*50)
    evaluate_acc(inf_list, val)

In [11]:
for i in range(len(infer_path)):
    evaluate_checkpoint(infer_path[i], False)
    evaluate_checkpoint(infer_path[i], True)

--------------------------------------------------
	 DeepSeek-R1-0528-Qwen3-8B
	 Evaluando: Test
--------------------------------------------------
              precision    recall  f1-score   support

          -1       0.00      0.00      0.00         3
           0       0.67      0.61      0.64        67
           1       0.80      0.67      0.73       105
           2       0.42      0.65      0.51        51

    accuracy                           0.64       226
   macro avg       0.47      0.48      0.47       226
weighted avg       0.67      0.64      0.64       226

[[ 0  2  0  1]
 [ 0 41 11 15]
 [ 0  6 70 29]
 [ 0 12  6 33]]
--------------------------------------------------
	 DeepSeek-R1-0528-Qwen3-8B
	 Evaluando: Validation
--------------------------------------------------
              precision    recall  f1-score   support

          -1       0.00      0.00      0.00         8
           0       0.71      0.76      0.73        58
           1       0.86      0.73      

/home/flopezp/.venvs/foo/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/flopezp/.venvs/foo/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/flopezp/.venvs/foo/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape

In [4]:
def get_labels(dataset, test):
    values = []
    for i in range(len(dataset)):
        text = dataset["Full"].iloc[i]
        text = re.sub('\n', '', text)
        unc = (text.count('Uncertain') + text.count('uncertain'), 'Uncertain')
        false = (text.count('False') + text.count('false'), 'False')
        true = (text.count('True') + text.count('true'), 'True')
        biggest = max(unc, false, true, key=lambda x:x[0])
        #print(f"Label: {biggest[1]}. T = {true[0]}, F= {false[0]}, U = {unc[0]}")
        if biggest[1] == 'True':
            values.append(1)
        elif biggest[1] == 'False':
            values.append(0)
        else:
            values.append(2)

    if test:
        print("------------------")
        print("====== TEST ======")
        print("------------------")
        print(classification_report(values, test_values))
        print(confusion_matrix(values, test_values))
    else:
        print("-----------------")
        print("====== VAL ======")
        print("-----------------")
        print(classification_report(values, val_values))
        print(confusion_matrix(values, val_values))

get_labels(llm_test_ds, True)
get_labels(llm_dev_ds, False)

------------------
====== TEST ======
------------------
              precision    recall  f1-score   support

           0       0.34      0.46      0.39        46
           1       0.87      0.48      0.62       158
           2       0.19      0.68      0.30        22

    accuracy                           0.50       226
   macro avg       0.47      0.54      0.44       226
weighted avg       0.70      0.50      0.54       226

[[21  8 17]
 [36 76 46]
 [ 4  3 15]]
-----------------
====== VAL ======
-----------------
              precision    recall  f1-score   support

           0       0.47      0.59      0.52        49
           1       0.83      0.48      0.61       126
           2       0.28      0.68      0.39        28

    accuracy                           0.53       203
   macro avg       0.53      0.58      0.51       203
weighted avg       0.67      0.53      0.56       203

[[29  7 13]
 [29 60 37]
 [ 4  5 19]]
